# 01 · Análisis Exploratorio (EDA)

**Parte II — Movement and Mental Health in Children**

Objetivo de este notebook: validar la estructura real de `datasets/raw/` antes de construir el ETL.

Comprobaciones clave:
1. Inventario de ficheros de sensor (`_F` fragmentos / `_T` por hora) por participante.
2. Autodetección de delimitador (F=`,`, T=`;`) y encoding UTF-8.
3. Fichero demográfico: confirmar ausencia de `SDQ19` y de `age`, y revisar nulos.
4. Estadísticas base de un fichero de sensor (columnas relevantes).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import etl_utils as eu
import pandas as pd

print('RAW_DIR:', eu.RAW_DIR)
print('¿existe?:', os.path.isdir(eu.RAW_DIR))

## 1. Inventario de ficheros de sensor por participante

In [ ]:
inv = eu.list_sensor_files()
inv_df = pd.DataFrame([
    {'participant_id': pid,
     'has_F': files['F'] is not None,
     'has_T': files['T'] is not None}
    for pid, files in inv.items()
])
print('Participantes con ficheros de sensor:', len(inv_df))
print('Con _F:', inv_df.has_F.sum(), '| Con _T:', inv_df.has_T.sum(),
      '| Con ambos:', (inv_df.has_F & inv_df.has_T).sum())
inv_df

## 2. Validación de delimitador y encoding

In [ ]:
# Tomamos el primer participante que tenga ambos ficheros y comprobamos el separador detectado.
sample = next((p for p, f in inv.items() if f['F'] and f['T']), None)
print('Participante de ejemplo:', sample)
for kind in ('F', 'T'):
    path = inv[sample][kind]
    print(f"  _{kind}: sep='{eu.detect_delimiter(path)}'  ->  {os.path.basename(path)}")

In [ ]:
# Leemos una muestra de cada tipo con el helper (solo columnas relevantes, renombradas).
df_F = eu.read_sensor_csv(inv[sample]['F']).head(1000)
df_T = eu.read_sensor_csv(inv[sample]['T']).head(1000)
print('Columnas (alias):', list(df_F.columns))
print('\n_F muestra:'); display(df_F.head(3))
print('_T muestra:'); display(df_T.head(3))

## 3. Fichero demográfico + cuestionarios

In [ ]:
demo = pd.read_csv(os.path.join(eu.RAW_DIR, eu.DEMOGRAPHIC_FILE), encoding='utf-8')
print('Participantes (filas):', len(demo))
print('Columnas:', list(demo.columns))

# Confirmaciones de las incidencias conocidas
print('\n¿SDQ19 presente?:', 'SDQ19' in demo.columns)
print('¿age presente?:', any(c.lower() == 'age' for c in demo.columns))
sdq_cols = sorted([c for c in demo.columns if c.startswith('SDQ')],
                  key=lambda c: int(c.replace('SDQ','')))
print('SDQ disponibles:', sdq_cols)

In [ ]:
# Nulos por columna (solo las que tengan algún ausente)
na = demo.isna().sum()
print('Columnas con valores ausentes:')
print(na[na > 0] if (na > 0).any() else '  (ninguna)')

# Resumen demográfico
print('\nSexo:'); print(demo['SEX'].value_counts(dropna=False))
demo[['BMI', 'height(cm)', 'weight(kg)']].describe()

## 4. Estadísticas base de un fichero de sensor

Comprobamos rangos de aceleración, pasos, distancia y batería, y las etiquetas de actividad.

In [ ]:
# Procesamos un _T por chunks para no cargarlo entero en memoria.
n_rows = 0
activities = {}
for chunk in eu.read_sensor_csv(inv[sample]['T'], chunksize=200_000):
    n_rows += len(chunk)
    vc = chunk['activity'].value_counts()
    for k, v in vc.items():
        activities[k] = activities.get(k, 0) + int(v)
print(f'Filas totales en {sample}_T:', n_rows)
print('Etiquetas de actividad:', activities)

In [ ]:
# Vista numérica de las primeras 50k filas de _T
df_stats = eu.read_sensor_csv(inv[sample]['T']).head(50_000)
df_stats[['acc_x','acc_y','acc_z','speed','steps','distance_m','battery']].describe()

## Conclusiones del EDA

- [ ] Nº de participantes y reparto F/T confirmado.
- [ ] Delimitadores correctos (F=`,`, T=`;`) y encoding UTF-8 sin mojibake.
- [ ] `SDQ19` y `age` ausentes confirmados → se tratan como nulos / prorrateo en scoring.
- [ ] Rango de variables de sensor razonable (sin valores centinela tipo -9999 sin filtrar).

> Siguiente paso: `02_etl_demographic.ipynb` (DimParticipant + scoring SDQ/SNAP → JSON).